# 1.2.5 SQL Basics for Data

SQLite3 in Python: CREATE, INSERT, SELECT, GROUP BY, JOIN, CTE — on Titanic data.

In [ ]:
import sqlite3, csv, urllib.request
from pathlib import Path
DATA=Path('data'); DATA.mkdir(exist_ok=True)
dest=DATA/'titanic.csv'
if not dest.exists():
    urllib.request.urlretrieve('https://huggingface.co/datasets/phihung/titanic/resolve/main/train.csv',dest)
print('CSV ready:', dest.stat().st_size, 'bytes')

In [ ]:
# Create DB + table
DB=DATA/'titanic.db'; DB.unlink(missing_ok=True)
conn=sqlite3.connect(DB)
conn.execute('''CREATE TABLE passengers(
    id INTEGER PRIMARY KEY, survived INTEGER, pclass INTEGER,
    name TEXT, sex TEXT, age REAL, fare REAL, embarked TEXT)''')
with open(dest,newline='') as f:
    rows=list(csv.DictReader(f))
records=[]
for r in rows:
    try: records.append((int(r['PassengerId']),int(r['Survived']),int(r['Pclass']),
        r['Name'],r['Sex'],float(r['Age'] or 0),float(r['Fare'] or 0),r.get('Embarked','')))
    except: pass
conn.executemany('INSERT INTO passengers VALUES (?,?,?,?,?,?,?,?)', records)
conn.commit()
print(f'Loaded {len(records)} rows')

In [ ]:
# SELECT + WHERE
for row in conn.execute('SELECT pclass, COUNT(*), ROUND(AVG(survived),3) FROM passengers GROUP BY pclass ORDER BY pclass'):
    print(f'Class {row[0]}: n={row[1]}, surv_rate={row[2]}')

In [ ]:
# GROUP BY + HAVING
for row in conn.execute("SELECT sex, COUNT(*), ROUND(AVG(survived),3) FROM passengers GROUP BY sex HAVING COUNT(*)>10"):
    print(f'{row[0]}: n={row[1]}, rate={row[2]}')

In [ ]:
# CTE
for row in conn.execute('''
    WITH stats AS (SELECT pclass, AVG(fare) as avg_fare FROM passengers GROUP BY pclass)
    SELECT p.name, p.fare, ROUND(p.fare - s.avg_fare, 2) as diff
    FROM passengers p JOIN stats s ON p.pclass=s.pclass
    WHERE p.survived=1 ORDER BY diff DESC LIMIT 5'''):
    print(row)